In [2]:
#This python script reads the two datasets, cleans them, and places them into a queryable dataset
import pandas as pd
import numpy as np

def load_data(qoi_path, years, attributes):
 
    #select columns that pertian to years/quantites of interest
    columns = []
    
    #regardless, we want the reigion and country name
    columns.append("country")
    columns.append("region")
    columns.append("iso3") #country 3-char code
    for year in years:
        for attribute in attributes:
            #following the naming convention of the file, each column header reads as
            column_header = str(attribute) + "_" + str(year) 
            columns.append(column_header)
            
    #give it the data path, as well as the years we are interested in.
    df = pd.read_csv(qoi_path, encoding="latin-1", usecols = columns)
    
    #replace all non-numeric data with np.nan, except for country and region
    cols_to_keep = ['country', 'region', "iso3"]
    numeric_cols = [col for col in df.columns if col not in cols_to_keep]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    
    #cut out any region codes--country codes have a decimal, so we delete all rows that have a decimal in the country code.
    df= df[~df['iso3'].str.contains(r'\.')]
    
    return df

def find_missing_data(df):
    
    #print countries that have missing data
    na_rows = df[df.isna().any(axis=1)]
    print("Countries with missing data:")
    print(na_rows["country"])
    
    #print number of nans per column
    print("\nNumber of nans per column:")
    print(df.isna().sum())
    
def query_by_country_code(df, code):
    country_row = df[df["iso3"] == code]
    return country_row.to_numpy()
    
               


In [3]:
years = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2023]
attributes = ["hdi", "le", "eys"]
df = load_data("data/HDR25_Composite_indices_complete_time_series.csv", years, attributes) 

#example use--get values for "GBR"
print(query_by_country_code(df, "GBR"))

#print number of nans by metric
print(find_missing_data(df))

df.to_csv("data/cleaned_metrics.csv", index=False, encoding='utf8')

[['GBR' 'United Kingdom' nan 0.812 0.835 0.87 0.903 0.921 0.931 0.93
  0.946 75.738 76.614 77.849 79.106 80.391 80.919 80.385 81.302
  13.64140034 14.85622501 15.99540997 16.61335945 16.45923996 17.32836914
  17.37801933 17.81106949]]
Countries with missing data:
3                  Andorra
4                   Angola
5      Antigua and Barbuda
8                Australia
9                  Austria
              ...         
179           Turkmenistan
185         United Kingdom
186          United States
188             Uzbekistan
189                Vanuatu
Name: country, Length: 93, dtype: object

Number of nans per column:
iso3         0
country      0
region      44
hdi_1990    54
hdi_1995    46
hdi_2000    24
hdi_2005     9
hdi_2010     4
hdi_2015     3
hdi_2020     3
hdi_2023     2
le_1990      0
le_1995      0
le_2000      0
le_2005      0
le_2010      0
le_2015      0
le_2020      0
le_2023      0
eys_1990    25
eys_1995    22
eys_2000    11
eys_2005     5
eys_2010     1
eys_2015  

In [4]:
print(df.shape)

(195, 27)


In [5]:
#Now, we want to load the immigration metrics between country.  Goal is to clean them and store in an adjaceny matrix, so that we can easily query by country, to country, by year
#chatGPT helped with this a good bit.
#we need the pycountry package here to convert country ISO3 code to numeric
import pycountry

iso3_to_numeric = {c.alpha_3: int(c.numeric) for c in pycountry.countries}
numeric_to_iso3_dict = {v: k for k, v in iso3_to_numeric.items()}

#numeric to iso
def numeric_to_iso3(num):
    return numeric_to_iso3_dict.get(num, None)  # returns None if code not found


In [ ]:
#load and the data: for now--we just load the data for all genders.  If we have time later, we can build in funcitonality to specify by gender:

#these are the columns that correspond to the ones that we want:
columns = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

#remove the first 9 columns
start_idx = 0

sheet_idx = 1

#NOTE: I had to manually remove the first few rows of this
df_migration = pd.read_excel('data/undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx', sheet_name=1)

#select only the columns we are interested in. 
columns = ['Index', 'Region, development group, country or area of destination', 'Location code of destination','Region, development group, country or area of origin', 'Location code of origin',
          1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024]

df_migration = df_migration[columns]


In [ ]:
#clean the data--i.e. drop all weird country codes

#the first thing we need to do is replace the country codes with ISO3 codes:
df_migration['destination code'] = df_migration['Location code of destination'].map(numeric_to_iso3)
df_migration['origin code'] = df_migration['Location code of origin'].map(numeric_to_iso3)

#drop all columns that do not have a corresponding country code 
df_migration = df_migration[
    df_migration['destination code'].notna() &
    df_migration['origin code'].notna()
]

In [44]:
df_migration.head()

,Index,"Region, development group, country or area of destination",Location code of destination,"Region, development group, country or area of origin",Location code of origin,1990,1995,2000,2005,2010,2015,2020,2024,destination code,origin code
0,1,World,900,World,900,153916063.0,163176002,174566152,192788721,221020392,250042020,275284032,304021813,None,None
1,2,World,900,Sub-Saharan Africa,1834,14124662.0,15183742,14584913,16004417,18243295,22763602,27134957,30661610,None,None
2,3,World,900,Northern Africa and Western Asia,1833,14986109.0,17216219,18728264,21198002,25429492,32508087,37196853,40529326,None,None
3,4,World,900,Central and Southern Asia,1831,30342957.0,27930630,30008559,32445580,39400759,46011893,48594959,53948417,None,None
4,5,World,900,Eastern and South-Eastern Asia,1832,14465509.0,17262816,20822011,24315849,30053666,34562856,38223520,41409235,None,None


In [ ]:
#drop all columns that are not shared between the two datasets:
migration_codes = df_migration['origin code'].unique()
metric_codes = df['iso3'].unique()
shared_codes = set(df_migration['origin code']).intersection(df['iso3'])

df_migration = df_migration[df_migration['origin code'].isin(shared_codes)]

df = df[df['iso3'].isin(shared_codes)]

#standardize empty entries to np.nan:

df_migration = df_migration.replace(r'^\s*$',np.nan, regex=True)
df_migration = df_migration.replace( [None, ''],np.nan)

df.to_csv("data/cleaned_metrics.csv", index=False, encoding='utf8')
df_migration.to_csv("data/cleaned_migration.csv",index = False, encoding = 'utf8')